# Calculating Summary Statistics of Pandas DataFrames

This notebook demonstrates how to compute summary statistics across entire Pandas DataFrames. Such statistics, like calculating an average, sometimes requires selecting only columns with numeric data types, so the notebook covers selecting columns by data type. It also covers applying custom functions to columns.

## Setup

In [1]:
import pandas as pd
import numpy as np

## Example DataFrame
We'll use an example dataset of student test scores.

In [2]:
df = pd.DataFrame({
    "student": ["Vivek", "Lee", "Jiwei", "Darryl"],
    "score_1": [100, 56, 88, 67],
    "score_2": [np.nan, 92, np.nan, 65],
    "score_3": [np.nan, np.nan, np.nan, 66],
    "attempts": [1, 2, 1, 3],
    "pass": [True, True, True, False]
})

df

,student,score_1,score_2,score_3,attempts,pass
0,Vivek,100,NaN,NaN,1,True
1,Lee,56,92.0,NaN,2,True
2,Jiwei,88,NaN,NaN,1,True
3,Darryl,67,65.0,66.0,3,False


## Column-wise Summary Statistics

### Mean
The `df.mean()` method attempts to compute the mean for each column. 
However, this is only possible for numeric columns (integers, floats, and booleans) in a DataFrame. 
If the DataFrame contains string or other non-numeric columns, Pandas will raise an error **unless**
`numeric_only=True` is specified.

The following code will throw an error due to the `name` column being a string data type.

In [3]:
df.mean()

TypeError: Could not convert ['VivekLeeJiweiDarryl'] to numeric

But including `numeric_only=True` will provide means for only the numeric columns in the DataFrame.

In [4]:
df.mean(numeric_only=True)

score_1     77.75
score_2     78.50
score_3     66.00
attempts     1.75
pass         0.75
dtype: float64

### Standard Deviation
For each column. This provides the unbiased estimator so it cannot calculate a deviation for the `score_3` column, which only has one non-NaN value.

In [5]:
df.std(numeric_only=True)

score_1     19.906029
score_2     19.091883
score_3           NaN
attempts     0.957427
pass         0.500000
dtype: float64

### Standard Error of the Mean (SEM)
SEM is calculated as standard deviation divided by the square root of the sample size.
Missing values are automatically ignored.

In [6]:
df.sem(numeric_only=True)

score_1      9.953015
score_2     13.500000
score_3           NaN
attempts     0.478714
pass         0.250000
dtype: float64

## Selecting Numeric Columns
Sometimes it's useful to simply select the numeric columns from a DataFrame before running summary statistics. The `select_dtypes('number')` method is useful to select just columns with numeric data types.

In [7]:
numeric_df = df.select_dtypes('number')
numeric_df

,score_1,score_2,score_3,attempts
0,100,NaN,NaN,1
1,56,92.0,NaN,2
2,88,NaN,NaN,1
3,67,65.0,66.0,3


## Row-wise Summary Statistics
Let's demonstrate taking summary statistics for every row with a dataset of Formula One races.

In [8]:
df_2 = pd.DataFrame({
    "race_1": [25, 12, 18, 1],
    "race_2": [25, 12, 18, 5],
    "race_3": [25, 12, 18, np.nan],
}, index =  ["Max Verstappen", "Lewis Hamilton", "Fernando Alonso", "Lando Norris"])

# Calculate the mean points of each driver over the 3 races.
df_2.mean(axis=1, numeric_only=True)

Max Verstappen     25.0
Lewis Hamilton     12.0
Fernando Alonso    18.0
Lando Norris        3.0
dtype: float64

In [9]:
df_2

,race_1,race_2,race_3
Max Verstappen,25,25,25.0
Lewis Hamilton,12,12,12.0
Fernando Alonso,18,18,18.0
Lando Norris,1,5,NaN


## Applying Functions to Columns
Default summary statistics like `.mean()` are great, but there are times when you want to apply your own particular transformation to each element in a column (or the index) of a DataFrame.

We'll illustrating applying your own transformations and functions with [`.map()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html) and [`.apply()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html).

### Using `.map()` to transform existing columnn
`df.col.map()` takes a function or dictionary and returns a transformation of each value in the column. Let's look at an example applying a dictionary where each key is mapped to the value output from the `team_map` dictionary.

In [10]:
team_map = {
    "Max Verstappen": "Red Bull Racing",
    "Lewis Hamilton": "Mercedes",
    "Fernando Alonso": "Aston Martin",
    "Lando Norris": "McLaren"
}

#map transforms a single column (or index). It accepts a Python function, a dict, or another Series.
df_2["Team"] = df_2.index.map(team_map)
df_2

,race_1,race_2,race_3,Team
Max Verstappen,25,25,25.0,Red Bull Racing
Lewis Hamilton,12,12,12.0,Mercedes
Fernando Alonso,18,18,18.0,Aston Martin
Lando Norris,1,5,NaN,McLaren


## Applying Custom Functions

You can also use `.map()` to apply a custom transformation to each element in a column based on a user-defined function. Each element in a column will be passed as input to the function. We'll use this functionality to automatically append an abbreviation to the team name.

In [11]:
def append_abbreviation(text):
    """ Append (ACRONYM) where (ACRONYM) is all capital letters in the input text in order """
    capitals = [character for character in text if character.isupper()]
    return text + " (" + "".join(capitals) + ")" # "".join(list) converts a list to a combined string

In [12]:
df_2["Team_with_acronym"] = df_2.Team.map(append_abbreviation)
df_2

,race_1,race_2,race_3,Team,Team_with_acronym
Max Verstappen,25,25,25.0,Red Bull Racing,Red Bull Racing (RBR)
Lewis Hamilton,12,12,12.0,Mercedes,Mercedes (M)
Fernando Alonso,18,18,18.0,Aston Martin,Aston Martin (AM)
Lando Norris,1,5,NaN,McLaren,McLaren (ML)


There are also times when you might want to do your own custom calculation on a dataframe for each row or each column. Here we'll write a function to count the number of races a driver finished (that are not NaN). The input to the function will be a row.

In [13]:
def count_races(x): #Function to Count the number of races a driver finished(Not NaN)
    return x.count()

The `.apply()` function allows you to apply a custom function **across columns or rows** of a DataFrame (`.map` is just for columns, but can accept dictionaries and a wider range of input than just functions).

It is especially useful when the calculation depends on multiple values within a row or column (`.map` can't do this).

To specify that you want a value returned for each column, pass `axis=1` to `.apply`:

In [14]:
df_2.select_dtypes("number").apply(count_races, axis = 1)

Max Verstappen     3
Lewis Hamilton     3
Fernando Alonso    3
Lando Norris       2
dtype: int64

### Practice question

Write a function that counts the number of races **each driver did not finish**  
(i.e., the number of `NaN` values in a row). Use the dataframe defined below.

In [15]:
df_3  = pd.DataFrame({
    "race_1": [25, np.nan, 18, 1],
    "race_2": [25, 12, np.nan, np.nan],
    "race_3": [25, 12, 18, np.nan]
}, index =  ["Max Verstappen", "Lewis Hamilton", "Fernando Alonso", "Lando Norris"])


In [16]:
df_3

,race_1,race_2,race_3
Max Verstappen,25.0,25.0,25.0
Lewis Hamilton,NaN,12.0,12.0
Fernando Alonso,18.0,NaN,18.0
Lando Norris,1.0,NaN,NaN


In [23]:
def race_not_finished(value):
    if pd.isna(value):
        return 1
    else: 
        return 0


In [24]:
df_3["race_1"].apply(race_not_finished)

Max Verstappen     0
Lewis Hamilton     1
Fernando Alonso    0
Lando Norris       0
Name: race_1, dtype: int64

In [25]:
df_3["race_2"].apply(race_not_finished)

Max Verstappen     0
Lewis Hamilton     0
Fernando Alonso    1
Lando Norris       1
Name: race_2, dtype: int64

In [26]:
df_3["race_3"].apply(race_not_finished)

Max Verstappen     0
Lewis Hamilton     0
Fernando Alonso    0
Lando Norris       1
Name: race_3, dtype: int64

## Summary

- Pandas can compute summary statistics across **entire DataFrames**.
- Methods like `df.mean()`, `df.std()`, and `df.sem()` operate column-wise by default.
- Use `numeric_only=True` to safely compute statistics when a DataFrame contains mixed data types.
- Row-wise summaries can be computed using `axis=1`.
- `df.select_dtypes('number')` is useful for isolating numeric columns before applying calculations.
- `.map()` is best for **one-to-one transformations** on a DataFrame column or index.
- `.apply()` is used when computations depend on **multiple values** within a row or column.
